In [51]:
import pandas as pd
import numpy as np
import math
import pickle
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import fbeta_score, precision_recall_curve, auc, confusion_matrix, precision_score, recall_score



In [19]:
df = pd.read_pickle("../data/clean.pkl")
display(df.head())

,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
0,1,298.1,308.6,1551,42.8,0,0,6952.0,0.0,10.0
1,0,298.2,308.7,1408,46.3,3,0,6827.0,139.0,10.0
2,0,298.1,308.5,1498,49.4,5,0,7749.0,247.0,10.0
3,0,298.2,308.6,1433,39.5,7,0,5928.0,276.0,10.0
4,0,298.2,308.7,1408,40.0,9,0,5898.0,360.0,10.0


In [ ]:
#split into two df on machine failure 0/1 MANUALLY

training_ratio = 0.65
testing_ratio = 1 - training_ratio

failures = df[df["machine_failure"] == 1]
normal = df[df["machine_failure"] == 0]

display(normal.head())
display(failures.head())

print(len(failures))

training_normal = normal.sample(frac=0.65, random_state=42)
testing_normal = normal.drop(training_normal.index)

training_failures = failures.sample(frac=0.65, random_state=42)
testing_failures = failures.drop(training_failures.index)

training = pd.concat([training_normal,training_failures] , ignore_index=True)
testing = pd.concat([testing_normal, testing_failures] , ignore_index=True)

print(f"Number of failures {training["machine_failure"].sum()} , percent of failures {training["machine_failure"].sum() / len(training)} in training")
print(f"Number of failures {testing["machine_failure"].sum()} , percent of failures {testing["machine_failure"].sum()/len(testing)} in testing")


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
0,1,298.1,308.6,1551,42.8,0,0,6952.0,0.0,10.0
1,0,298.2,308.7,1408,46.3,3,0,6827.0,139.0,10.0
2,0,298.1,308.5,1498,49.4,5,0,7749.0,247.0,10.0
3,0,298.2,308.6,1433,39.5,7,0,5928.0,276.0,10.0
4,0,298.2,308.7,1408,40.0,9,0,5898.0,360.0,10.0


,type,air_temperature,process_temperature,rotational_speed,torque,tool_wear,machine_failure,power,overstrain,temperature_difference
50,0,298.9,309.1,2861,4.6,143,1,1378.0,658.0,10.0
69,0,298.9,309.0,1410,65.7,191,1,9701.0,12549.0,10.0
77,0,298.8,308.9,1455,41.3,208,1,6293.0,8590.0,10.0
160,0,298.4,308.2,1282,60.7,216,1,8149.0,13111.0,10.0
161,0,298.3,308.1,1412,52.3,218,1,7733.0,11401.0,10.0


339
Number of failures 220 , percent of failures 0.033846153846153845 in training
Number of failures 119 , percent of failures 0.034 in testing


Evaluation strategy:

Primary metric: F2 score, recall weighted more heavily than precison as missed failures are more costly than false alarms
Secondary metric: PR-AUC, to assess performance across all thresholds not just at 0.5
Supporting data:
- Confusion matrix
- Cross-validation with stratifiedKfold with k = 5. Splitting training data into 5 chunks training using 4 of them and testing on the 5th repeat until all of the 5 chunks have been the mock test data 
Thus the above is just an exmaple of how data could be split. But I will divide it into 5 chunks and construct data frames for training and testing using them. Will move from 65% training data as it is awkward to do with the chunks.


In [ ]:
#Split using sklearn

X = df.drop(columns=["machine_failure"])
y = df["machine_failure"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))


machine_failure
0    0.966125
1    0.033875
Name: proportion, dtype: float64
machine_failure
0    0.966
1    0.034
Name: proportion, dtype: float64


Benchmarking

In [55]:
def evaluation_metrics(y_test,y_pred,y_prob= None):
    #calc recall
    #calc precision
    #compute fbeta, confusion matrix, pr-uca
    confus_mar = confusion_matrix(y_test, y_pred)
    f2 = fbeta_score(y_test,y_pred, beta = 2)
    precision = precision_score(y_test,y_pred)
    recall = recall_score(y_test,y_pred)

    pr_auc = None
    if y_prob is not None:
        pre, re, _ = precision_recall_curve(y_test, y_prob)
        pr_auc = auc(re,pre)

    return {"Confusion Matrix": confus_mar, 
            "f2":f2,
            "precision": precision,
            "recall" : recall, 
            "pr auc": pr_auc}




In [60]:
#Dummy classifier


dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train,y_train)

y_pred = dummy.predict(X_test)
probs = dummy.predict_proba(X_test)[:,1]
y_actual = sum(y_test)
print(sum(y_pred))
print(sum(y_test))

fbeta_score(y_test, y_pred, beta = 2)

eval_dum = evaluation_metrics(y_test, y_pred, probs)
print(eval_dum)


0
68
{'Confusion Matrix': array([[1932,    0],
       [  68,    0]]), 'f2': 0.0, 'precision': 0.0, 'recall': 0.0, 'pr auc': 0.517}


c:\Users\tbarn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
#Logistic Regression, using 0.5 probability threshold, UNBALANCED

model = LogisticRegression(random_state=42)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(f"{y_pred} \n {sum(y_pred)}")
print(f"{sum(y_test)}")
probs = model.predict_proba(X_test)[:,1]
y_test_list = list(y_test)


eval_lru = evaluation_metrics(y_test, y_pred, probs)
print(eval_lru)

[1 0 0 ... 0 0 0] 
 23
68
{'Confusion Matrix': array([[1923,    9],
       [  54,   14]]), 'f2': 0.23728813559322035, 'precision': 0.6086956521739131, 'recall': 0.20588235294117646, 'pr auc': 0.49525591106226746}


c:\Users\tbarn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [57]:
#Logistic Regression, using 0.5 probability threshold, BALANCED

model = LogisticRegression(class_weight="balanced",random_state=42)
model.fit(X_train,y_train)

y_pred = model.predict(X_test)
print(f"{y_pred} \n {sum(y_pred)}")
print(f"{sum(y_test)}")
probs = model.predict_proba(X_test)[:,1]
y_test_list = list(y_test)


eval_lrb = evaluation_metrics(y_test, y_pred, probs)
print(eval_lrb)

[1 0 0 ... 0 1 0] 
 344
68
{'Confusion Matrix': array([[1648,  284],
       [   8,   60]]), 'f2': 0.487012987012987, 'precision': 0.1744186046511628, 'recall': 0.8823529411764706, 'pr auc': 0.4386394891351097}


c:\Users\tbarn\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
